# Dynamics Transformer — perturbation × time → per-gene response

Physics-informed gene-transformer trained on **LINCS L1000** to learn how genes respond to perturbations over time, held out **by perturbation** (unseen perturbagens at test).

**Needs a GPU runtime**: Runtime → Change runtime type → GPU (L4/T4/A100). ~5 GB download + ~15–40 min train.

Pipeline: physics features (half-lives) → LINCS training table → train + held-out eval → save checkpoint to Drive.

In [ ]:
# 1) GPU check + clone repo + deps
import torch, subprocess, os, sys
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE  ->  Runtime > Change runtime type > GPU')
if not os.path.isdir('/content/cell'):
    subprocess.run('git clone -b claude/vectorize-gex-propensity-NRqBW https://github.com/nikku03/cell.git /content/cell', shell=True)
os.chdir('/content/cell'); subprocess.run(['git','pull'])
subprocess.run('pip install -q cmapPy h5py pandas openpyxl', shell=True)
print('cwd', os.getcwd())

In [ ]:
# 2) PHYSICS features: measured mRNA + protein half-lives (Mathieson / RNADecayCafe, CC-BY)
import subprocess, sys
r=subprocess.run([sys.executable,'colab/fetch_dynamics_data.py'],capture_output=True,text=True)
print(r.stdout[-500:]); print('ERR',r.stderr[-400:]) if r.stderr.strip() else None

In [ ]:
# 3) LINCS L1000 -> training table  (GSE70138 = 5 GB; set GSE92742 for the full 20 GB)
import subprocess, sys, os
os.environ['LINCS_GSE']='GSE70138'        # phase 2 (smaller); 'GSE92742' = phase 1 (bigger, more perts)
os.environ['LINCS_MAX_SIGS']='120000'     # cap signatures to keep RAM/time sane; raise on A100
r=subprocess.run([sys.executable,'colab/fetch_lincs.py'],capture_output=True,text=True)
print(r.stdout[-1600:]); print('ERR',r.stderr[-900:]) if r.stderr.strip() else None

In [ ]:
# 4) TRAIN the dynamics transformer (held out by perturbation) + eval vs baselines
import subprocess, sys, os
os.environ['DTF_EPOCHS']='15'; os.environ['DTF_DIM']='128'; os.environ['DTF_LAYERS']='3'; os.environ['DTF_BATCH']='256'
r=subprocess.run([sys.executable,'colab/dynamics_transformer.py'],capture_output=True,text=True)
print(r.stdout[-2200:]); print('ERR',r.stderr[-900:]) if r.stderr.strip() else None

In [ ]:
# 5) save checkpoint + eval to Drive (so a future runtime reuses it)
from google.colab import drive; drive.mount('/content/drive')
import shutil, os, json
d='/content/drive/MyDrive/virtual_cell_data/dynamics_transformer'; os.makedirs(d,exist_ok=True)
for f in ['dynamics_transformer.pt','dynamics_transformer_eval.json','lincs_train.npz']:
    p=f'outputs/orphan/{f}'
    if os.path.exists(p): shutil.copy(p,d); print('saved',f, os.path.getsize(p)//1048576,'MB')
print('\neval:', json.load(open('outputs/orphan/dynamics_transformer_eval.json')) if os.path.exists('outputs/orphan/dynamics_transformer_eval.json') else 'no eval file')